# 00: 環境と観測の確認

カメラ画像、8次元の状態、7次元の行動を確認します。依存ライブラリは実行時に取得します。

1. Google Driveの「マイドライブ」に `vla_Workshop` フォルダを作り、その中に `00_environment` をフォルダごと置きます。
2. このノートブックをColabで開き、ランタイムを **T4 GPU** にします。別の場所に置いた場合は、最初のコードセルの `LESSON_DIR` を変更します。
3. 上から順にセルを実行します。

同じフォルダに置くファイル

- `lift.json`
- `lift_tools.py`
- `prepare.py`
- `requirements.txt`

In [ ]:
# @title この回の配布ファイルを準備する
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_Workshop/00_environment"  # @param {type:"string"}
REQUIRED_FILES = [
    "lift.json",
    "lift_tools.py",
    "prepare.py",
    "requirements.txt",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_00")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

次のセルで、この回に必要なライブラリを準備します。初回は数分かかることがあります。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py")]

subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## Liftを起動して、グリッパーを開く

Liftは、ロボットアームで立方体を持ち上げる練習用の環境です。手先を同じ位置に保ってグリッパーを開き、カメラ画像と動画を保存します。

In [ ]:
result = run_python(r"""
import json
import os
import tempfile
from pathlib import Path

import imageio.v2 as imageio
import numpy as np

# 描画バックエンドをrobosuiteのimportより先に設定します。
settings = json.loads(Path("lift.json").read_text())
config = settings["env"]
os.environ["MUJOCO_GL"] = config["renderer"]
os.environ["PYOPENGL_PLATFORM"] = config["renderer"]
import robosuite
from robosuite import macros
from lift_tools import lift_controller, state_vector

macros.IMAGE_CONVENTION = "opengl"
env = robosuite.make(
    env_name=config["task"],
    robots=config["robot"],
    controller_configs=lift_controller(),
    has_renderer=False,
    has_offscreen_renderer=True,
    use_camera_obs=True,
    use_object_obs=False,
    camera_names=config["cameras"],
    camera_heights=config["image_size"],
    camera_widths=config["image_size"],
    control_freq=config["control_freq"],
    horizon=settings["max_steps"] + config["settle_steps"],
    hard_reset=False,
    reward_shaping=False,
    seed=settings["seed"],
)
try:
    obs = env.reset()
    for _ in range(config["settle_steps"]):
        obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])

    folder = Path(tempfile.mkdtemp(prefix="lift-", dir="outputs")).resolve()
    camera_key = f'{config["cameras"][0]}_image'
    imageio.imwrite(folder / "observation.png", obs[camera_key][::-1])
    steps = []
    with imageio.get_writer(folder / "video.mp4", fps=config["control_freq"]) as video:
        video.append_data(obs[camera_key][::-1])
        for step in range(settings["max_steps"]):
            state = state_vector(obs)
            # 位置・姿勢はそのままで、グリッパーを開きます。
            action = np.array([0, 0, 0, 0, 0, 0, -1], dtype=np.float32)
            obs, reward, done, _ = env.step(action)
            video.append_data(obs[camera_key][::-1])
            steps.append(
                {
                    "step": step,
                    "state_before": state.tolist(),
                    "action": action.tolist(),
                    "reward": reward,
                }
            )
            if done or reward > 0:
                break
    report = {
        "camera_shape": list(obs[camera_key].shape),
        "state_order": ["x", "y", "z", "rx", "ry", "rz", "gripper_left", "gripper_right"],
        "action_order": ["dx", "dy", "dz", "drx", "dry", "drz", "gripper"],
        "steps": steps,
    }
    (folder / "observations.json").write_text(json.dumps(report, indent=2))
    print(f"Results: {folder}")
finally:
    env.close()
""")

## 画像・動画・仕様を確認する

In [ ]:
import json

from IPython.display import Image, Video, display

display(Image(filename=str(result / "observation.png")))
display(Video(str(result / "video.mp4"), embed=True))
report = json.loads((result / "observations.json").read_text())
print("カメラ画像の形:", report["camera_shape"])
print("状態の順番:", report["state_order"])
print("行動の順番:", report["action_order"])
print("最初のステップ:", report["steps"][0])

## 確認課題

出力の「状態の順番」と「行動の順番」を見て、位置、向き、グリッパーに対応する値を確認してください。

## 結果をDriveに保存する

観測画像、動画、状態・行動ログを保存します。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_Workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = "00_environment_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)